# Notebook for transcribing audio using Google Cloud Speech-to-Text

### imports

In [1]:
import quail
import os
import shutil
import pickle
import re
import pandas as pd
from nltk.corpus import stopwords

## create speech context from union of words in annotations

In [2]:
atlep1_df = pd.read_pickle('../../data/annotations_dfs/atlep1.p')
atlep2_df = pd.read_pickle('../../data/annotations_dfs/atlep2.p')
arrdev_df = pd.read_pickle('../../data/annotations_dfs/arrdev.p')

In [3]:
def meets_word_criteria(string):
    """
    Removes words with characters not wanted in auto transcriber speech context
    """
    
    good_word = True
    
    # remove words containing digits
    if any(char.isdigit() for char in string):
        good_word = False
    
    # remove words surrounded by single quotes and possessives (avoid duplicates in nested quotations & possessives)
    if string.startswith("'") or string.endswith("'") or string.endswith("'s"):
        good_word = False
        
    # remove unhelpful simple words
    if len(string) <= 2:
        good_word = False
    
    return good_word

In [4]:
def create_speech_context(df):
    """
    Creates episode-specific speech context from video annotations
    """
    
    # use Narrative details (internal and external), Characters on screen, Speech, Character speaking, and Setting
    word_cols = [df.columns[i] for i in [2,3,4,6,7,9]]
    
    # create single string of all text
    allwords = ' '.join(df.loc[:,word_cols].apply(lambda x: ' '.join(x.dropna()), axis=1).values.tolist())
    
    # remove all characters except spaces (catches \n and \t), letters, apostrophes, dashes
    no_punctuation = re.sub("[^\w\s'-]+", '', allwords)
    
    speech_context = []
    
    # split words into list
    for word in no_punctuation.split():
        # identify unique words that meet criteria
        if word.upper() not in speech_context and meets_word_criteria(word):
            speech_context.append(word.upper())

    # remove English stopwords
    speech_context_nostop = [word for word in speech_context if word not in stopwords.words('english')]
    
    return speech_context_nostop

In [5]:
atlep1_speech_context = create_speech_context(atlep1_df)
atlep2_speech_context = create_speech_context(atlep2_df)
arrdev_speech_context = create_speech_context(arrdev_df)

## load in experiment data and mapings between subject ID & PsiTurk ID

In [6]:
with open('../../data/pickles/expdf.p', 'rb') as f:
    expdf = pickle.load(f)

with open('../../data/pickles/id_maps.p', 'rb') as f:
    id_maps = pickle.load(f)

## set some paths

In [7]:
audiodir = os.path.abspath('../../data/audio/')
transcdir = os.path.abspath('../../data/transcriptions/automatic/')
keypath = os.path.abspath('../../../google-credentials/cloud-speech-credentials.json')

## create directory structure

In [8]:
for sid, data in id_maps.items():
    for ses, turkid in data.items():
        folder = os.path.join(transcdir,sid,turkid)
        if not os.path.isdir(folder) and not os.path.isdir(os.path.join(transcdir,'drops',sid)):
            os.makedirs(folder)

## transcribe all audio files not previously transcribed

In [9]:
turkids = [tid for l in [list(ses.values()) for ses in id_maps.values()] for tid in l if tid]

done = False

# walk audio folder
for root, dirs, files in os.walk(audiodir):
    
    # ignore parent dirs with hiden files
    if [f for f in files if not f.startswith('.')]:
        # assign psiturk id
        turkid = files[0].split('-')[0]
        # ignore drops
        if turkid in turkids:
            # assign subject id
            sid = expdf.loc[expdf.uniqueid==turkid]['Subject ID'].values[0]
            audio_files = [file for file in files if file.endswith('wav')]
            
            for audio_file in audio_files:
                
                # set correct speech context
                if (any([af.split('-')[1].startswith('prediction') for af in audio_files]) 
                    or audio_file.split('-')[1].startswith('delayed')):
                    speech_context = atlep1_speech_context
                elif 'A' in sid:
                    speech_context = atlep2_speech_context
                else:
                    speech_context = arrdev_speech_context
                
                af_path = os.path.join(root,audio_file)
                save_dir = os.path.join(transcdir,sid,turkid)
                
                # skip over previously decoded audio
                if not os.path.isfile(os.path.join(save_dir,audio_file+'.txt')):
                    
                # decode audio file and save in specified dir
                    print('decoding ' + audio_file)
                    quail.decode_speech(af_path, keypath=keypath, save=True, save_dir=save_dir,
                                        speech_context=speech_context, max_alternatives=5)
                
                else:
                    print('already finished '+ audio_file)

already finished debugvnS2Q:debugG20gK-prediction.wav
already finished debugvnS2Q:debugG20gK-recall.wav
already finished debugRNwge:debugv0fyi-recall.wav
already finished debugRNwge:debugv0fyi-prediction.wav
already finished debugpdN5k:debug3flgN-recall.wav
already finished debugpdN5k:debug3flgN-prediction.wav
already finished debugWvhTI:debug65Thd-recall.wav
already finished debugWvhTI:debug65Thd-prediction.wav
already finished debug3dOrm:debugAjPUS-prediction.wav
already finished debug3dOrm:debugAjPUS-recall.wav
already finished debugfO0us:debug6tlz1-recall.wav
already finished debugfO0us:debug6tlz1-prediction.wav
already finished debugLou4K:debug5KpMQ-prediction.wav
already finished debugLou4K:debug5KpMQ-recall.wav
already finished debugJ2iI9:debugD6tOV-delayed.wav
already finished debugJ2iI9:debugD6tOV-recall.wav
already finished debugNTcAd:debugiaabY-delayed.wav
already finished debugNTcAd:debugiaabY-recall.wav
already finished debuglAycr:debugoFV0D-recall.wav
already finished deb

Audio clip is longer than 1 minute.  Splitting into 14 one minute segments...
Transcript: so I think that episode
Confidence: 0.9829309582710266
Word: so, start_time: 2.1, end_time: 2.5
Word: I, start_time: 2.5, end_time: 2.6
Word: think, start_time: 2.6, end_time: 2.8
Word: that, start_time: 2.8, end_time: 2.9
Word: episode, start_time: 2.9, end_time: 4.0
Transcript:  started out when they were on the boat and it basically like goes through introducing
Confidence: 0.9370238184928894
Word: started, start_time: 5.0, end_time: 5.8
Word: out, start_time: 5.8, end_time: 5.9
Word: when, start_time: 5.9, end_time: 8.5
Word: they, start_time: 8.5, end_time: 8.6
Word: were, start_time: 8.6, end_time: 8.8
Word: on, start_time: 8.8, end_time: 9.1
Word: the, start_time: 9.1, end_time: 9.2
Word: boat, start_time: 9.2, end_time: 9.4
Word: and, start_time: 9.4, end_time: 10.4
Word: it, start_time: 10.4, end_time: 10.5
Word: basically, start_time: 10.5, end_time: 10.7
Word: like, start_time: 10.7, en

Transcript:  thinks that like a husband and wife can't be arrested for the same crime
Confidence: 0.9765916466712952
Word: thinks, start_time: 12.1, end_time: 12.8
Word: that, start_time: 12.8, end_time: 13.1
Word: like, start_time: 13.1, end_time: 13.2
Word: a, start_time: 13.2, end_time: 13.5
Word: husband, start_time: 13.5, end_time: 13.8
Word: and, start_time: 13.8, end_time: 14.0
Word: wife, start_time: 14.0, end_time: 14.0
Word: can't, start_time: 14.0, end_time: 14.6
Word: be, start_time: 14.6, end_time: 14.6
Word: arrested, start_time: 14.6, end_time: 15.0
Word: for, start_time: 15.0, end_time: 15.1
Word: the, start_time: 15.1, end_time: 15.2
Word: same, start_time: 15.2, end_time: 15.2
Word: crime, start_time: 15.2, end_time: 15.9
Transcript:  so that Michael like is returns to the house and it's like getting everything packed up and here's his son talk about how like he wishes that he could spend more time with the family so he feels bad board game ends
Confidence: 0.94270575

Audio clip is longer than 1 minute.  Splitting into 14 one minute segments...
Transcript: the episode began with paper boy and his cousin I don't remember his name so we'll just call him
Confidence: 0.9314562678337097
Word: the, start_time: 0.7000000000000001, end_time: 1.1
Word: episode, start_time: 1.1, end_time: 1.8
Word: began, start_time: 1.8, end_time: 2.1
Word: with, start_time: 2.1, end_time: 2.7
Word: paper, start_time: 2.7, end_time: 3.7
Word: boy, start_time: 3.7, end_time: 3.8
Word: and, start_time: 3.8, end_time: 4.4
Word: his, start_time: 4.4, end_time: 4.5
Word: cousin, start_time: 4.5, end_time: 5.1
Word: I, start_time: 5.1, end_time: 6.2
Word: don't, start_time: 6.2, end_time: 6.7
Word: remember, start_time: 6.7, end_time: 7.0
Word: his, start_time: 7.0, end_time: 7.2
Word: name, start_time: 7.2, end_time: 7.3
Word: so, start_time: 7.3, end_time: 7.8
Word: we'll, start_time: 7.8, end_time: 8.4
Word: just, start_time: 8.4, end_time: 8.4
Word: call, start_time: 8.4, end_

Word: and, start_time: 5.3, end_time: 5.5
Word: she, start_time: 5.5, end_time: 5.9
Word: says, start_time: 5.9, end_time: 6.2
Finished file 1 of 1 in 442.33 seconds.
decoding debug5T48Z:debugbTZe9-recall.wav
Decoding file 1 of 1
Audio clip is longer than 1 minute.  Splitting into 16 one minute segments...
Transcript: the episode begins with George Michael on the ship
Confidence: 0.9284638166427612
Word: the, start_time: 1.3, end_time: 2.0
Word: episode, start_time: 2.0, end_time: 2.7
Word: begins, start_time: 2.7, end_time: 3.2
Word: with, start_time: 3.2, end_time: 3.6
Word: George, start_time: 3.6, end_time: 4.4
Word: Michael, start_time: 4.4, end_time: 4.9
Word: on, start_time: 4.9, end_time: 5.6
Word: the, start_time: 5.6, end_time: 5.7
Word: ship, start_time: 5.7, end_time: 6.0
Transcript:  narrator who is
Confidence: 0.9343599081039429
Word: narrator, start_time: 7.4, end_time: 8.6
Word: who, start_time: 8.6, end_time: 8.9
Word: is, start_time: 8.9, end_time: 9.2
Transcript:  na

Word: he, start_time: 14.1, end_time: 14.3
Word: pulls, start_time: 14.3, end_time: 14.6
Word: out, start_time: 14.6, end_time: 14.6
Word: his, start_time: 14.6, end_time: 14.8
Word: cartography, start_time: 14.8, end_time: 15.1
Word: mass, start_time: 15.1, end_time: 15.8
Word: and, start_time: 15.8, end_time: 16.4
Word: he, start_time: 16.4, end_time: 16.5
Word: said, start_time: 16.5, end_time: 16.6
Word: I, start_time: 16.6, end_time: 16.7
Word: was, start_time: 16.7, end_time: 16.8
Word: you, start_time: 16.8, end_time: 17.0
Word: the, start_time: 17.0, end_time: 17.1
Word: blue, start_time: 17.1, end_time: 17.3
Word: part, start_time: 17.3, end_time: 17.6
Word: is, start_time: 17.6, end_time: 17.7
Word: black, start_time: 17.7, end_time: 18.0
Transcript:  singing cats too
Confidence: 0.7723993062973022
Word: singing, start_time: 20.1, end_time: 21.2
Word: cats, start_time: 21.2, end_time: 21.6
Word: too, start_time: 21.6, end_time: 21.9
Transcript:  where are the flutes are plast